In [8]:
import numpy as np
import pandas as pd
import ast
import os
import re

# =====================================================
# CONFIG
# =====================================================
FILES = [
    "cpd_kappa_experiment_old.csv",
    "cpd_kappa_experiment_new.csv"
]

TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# parse est_CP（完全照搬）
# =====================================================
def parse_cp(x):
    if pd.isna(x):
        return []

    s = str(x).strip()

    if s in ["", "nan", "None"]:
        return []

    s = re.sub(r"np\.int64\((\d+)\)", r"\1", s)

    if "[" in s and "]" in s and "," not in s:
        s = s.replace(" ", ",")

    try:
        return list(ast.literal_eval(s))
    except:
        return []


# =====================================================
# evaluation（完全不动）
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)

    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.inf
        De_t = np.inf
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    CE = abs(len(est_cp) - len(true_cp))

    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# LOAD FILES
# =====================================================
df_list = []

for f in FILES:
    if not os.path.exists(f):
        continue

    tmp = pd.read_csv(f)
    tmp["source"] = "old" if "old" in f else "new"
    tmp["est_CP"] = tmp["est_CP"].apply(parse_cp)

    df_list.append(tmp)

df = pd.concat(df_list, ignore_index=True)

df["kappa"] = df["kappa"].astype(float)


# =====================================================
# COMPUTE METRICS
# =====================================================
df = df.drop(columns=["CE"], errors="ignore")

metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)
df["rho"] = df["kappa"]
df = df.drop(columns=["kappa"])

summary = (
    df.groupby(["rho"])[["FP","FN","Dt_e","De_t","CE","CS"]]
    .mean()
    .round(2)
    .reset_index()
    .sort_values("rho")
)

print("\n===== SUMMARY =====\n")
print(summary)


# =====================================================
# LATEX TABLE（并列 old / new）
# =====================================================
def to_latex_table(summary):

    latex = []
    latex.append("\\begin{table}[!ht]")
    latex.append("\\caption{Sensitivity analysis with respect to $\\rho$.}")
    latex.append("\\label{tab:rho}")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append("\\begin{tabular}{cccccccccc}")
    latex.append("\\toprule")
    latex.append("$\\rho$ & FP$\\downarrow$ & FN$\\downarrow$ & $D_{t\\to e}$ & $D_{e\\to t}$ & CE$\\downarrow$ & CS$\\uparrow$ \\\\")
    latex.append("\\midrule")

    for _, r in summary.iterrows():
        latex.append(
            f"{r['rho']} & "
            f"{r['FP']:.2f} & {r['FN']:.2f} & "
            f"{r['Dt_e']:.2f} & {r['De_t']:.2f} & "
            f"{r['CE']:.2f} & {r['CS']:.2f} \\\\"
        )

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)


latex_table = to_latex_table(summary)

print("\n===== LaTeX Table =====\n")
print(latex_table)


===== SUMMARY =====

   rho   FP   FN  Dt_e  De_t   CE    CS
0  0.2  0.3  0.0   0.0  12.6  0.3  0.96
1  0.5  0.8  0.6  34.9  36.6  0.6  0.80
2  0.8  1.0  0.0   0.2  61.9  1.0  0.90
3  1.5  0.7  0.4  27.1  42.0  0.5  0.85
4  2.0  1.0  0.4  32.4  65.4  0.8  0.79

===== LaTeX Table =====

\begin{table}[!ht]
\caption{Sensitivity analysis with respect to $\rho$.}
\label{tab:rho}
\centering
\small
\begin{tabular}{cccccccccc}
\toprule
$\rho$ & FP$\downarrow$ & FN$\downarrow$ & $D_{t\to e}$ & $D_{e\to t}$ & CE$\downarrow$ & CS$\uparrow$ \\
\midrule
0.2 & 0.30 & 0.00 & 0.00 & 12.60 & 0.30 & 0.96 \\
0.5 & 0.80 & 0.60 & 34.90 & 36.60 & 0.60 & 0.80 \\
0.8 & 1.00 & 0.00 & 0.20 & 61.90 & 1.00 & 0.90 \\
1.5 & 0.70 & 0.40 & 27.10 & 42.00 & 0.50 & 0.85 \\
2.0 & 1.00 & 0.40 & 32.40 & 65.40 & 0.80 & 0.79 \\
\bottomrule
\end{tabular}
\end{table}


In [1]:
import numpy as np
import pandas as pd
import re
import ast

# =====================================================
# CONFIG
# =====================================================
CSV_PATH = "cpd_kappa_experiment.csv"
TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# parse est_CP
# =====================================================
def parse_cp(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if pd.isna(x):
        return []
    try:
        return list(ast.literal_eval(str(x)))
    except:
        return []


# =====================================================
# evaluation
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)

    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.nan 
        De_t = np.nan
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    CE = abs(len(est_cp) - len(true_cp))

    # segmentation score
    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# LOAD
# =====================================================
df = pd.read_csv(CSV_PATH)

df["est_CP"] = df["est_CP"].apply(parse_cp)
df = df.drop(columns=["CE"], errors="ignore")


# =====================================================
# COMPUTE METRICS
# =====================================================
metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)


# =====================================================
# SUMMARY
# =====================================================
summary = (
    df.groupby("kappa")[["FP","FN","Dt_e","De_t","CE","CS"]]
    .mean()
    .round(2)
    .reset_index()
)

print(summary)


# =====================================================
# LATEX TABLE
# =====================================================
def to_latex_table(summary):
    rows = []

    best_FP = summary["FP"].min()
    best_FN = summary["FN"].min()
    best_Dte = summary["Dt_e"].min()
    best_Det = summary["De_t"].min()
    best_CE = summary["CE"].min()
    best_CS = summary["CS"].max()

    for _, r in summary.iterrows():

        def bold(val, best):
            if pd.notna(val) and val == best:
                return f"\\textbf{{{val:.2f}}}"
            return f"{val:.2f}"

        row = [
            r["kappa"],
            bold(r["FP"], best_FP),
            bold(r["FN"], best_FN),
            bold(r["Dt_e"], best_Dte),
            bold(r["De_t"], best_Det),
            bold(r["CE"], best_CE),
            bold(r["CS"], best_CS),
        ]
        rows.append(row)

    latex = []
    latex.append("\\begin{table}[!ht]")
    latex.append("\\caption{Sensitivity analysis with respect to $\\rho$ (kappa).}")
    latex.append("\\label{tab:kappa}")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append("\\begin{tabular}{cccccccc}")
    latex.append("\\toprule")
    latex.append("$\\rho$ & FP$\\downarrow$ & FN$\\downarrow$ & $D_{t\\to e}$ & $D_{e\\to t}$ & CE$\\downarrow$ & CS$\\uparrow$ \\\\")
    latex.append("\\midrule")

    for r in rows:
        latex.append(f"{r[0]:.2f} & {r[1]} & {r[2]} & {r[3]} & {r[4]} & {r[5]} & {r[6]} \\\\")

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)


latex_table = to_latex_table(summary)

print("\n===== LaTeX Table =====\n")
print(latex_table)

   kappa   FP   FN  Dt_e  De_t   CE    CS
0    0.5  0.8  0.6  34.9  36.6  0.6  0.80
1    0.8  1.0  0.0   0.2  61.9  1.0  0.90
2    2.0  1.0  0.4  32.4  65.4  0.8  0.79

===== LaTeX Table =====

\begin{table}[!ht]
\caption{Sensitivity analysis with respect to $\rho$ (kappa).}
\label{tab:kappa}
\centering
\small
\begin{tabular}{cccccccc}
\toprule
$\rho$ & FP$\downarrow$ & FN$\downarrow$ & $D_{t\to e}$ & $D_{e\to t}$ & CE$\downarrow$ & CS$\uparrow$ \\
\midrule
0.50 & \textbf{0.80} & 0.60 & 34.90 & \textbf{36.60} & \textbf{0.60} & 0.80 \\
0.80 & 1.00 & \textbf{0.00} & \textbf{0.20} & 61.90 & 1.00 & \textbf{0.90} \\
2.00 & 1.00 & 0.40 & 32.40 & 65.40 & 0.80 & 0.79 \\
\bottomrule
\end{tabular}
\end{table}
